### Convert data production coffee to CSV

In [ ]:
import pandas as pd
import os
import glob

def extract_year_data(filepath):
    year = int(os.path.basename(filepath).replace('.xlsx', ''))
    df = pd.read_excel(filepath, header=None)

    # Filter only actual village data rows (column 4 == 'Kopi Arabika')
    data = df[df[4] == 'Kopi Arabika'].copy()

    result = pd.DataFrame({
        'year':        year,
        'village':     data[3].str.strip(),
        'tm_ha':       data[6],
        'produksi_kg': data[10]
    })

    return result

# Process all xlsx files in the same folder as this script
files = sorted(glob.glob('../data/*.xlsx'))
print(f"Found files: {files}")

all_data = pd.concat([extract_year_data(f) for f in files], ignore_index=True)

all_data.to_csv('../data/coffeeProduction_benerMeriah.csv', index=False)

print(f"Total records : {len(all_data)}")
print(f"Years         : {sorted(all_data['year'].unique())}")
print(f"Villages      : {all_data['village'].nunique()} unique")
print(all_data.head(10))

Found files: ['../data/2021.xlsx', '../data/2022.xlsx', '../data/2023.xlsx', '../data/2024.xlsx', '../data/2025.xlsx']
Total records : 1100
Years         : [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Villages      : 219 unique
   year           village   tm_ha produksi_kg
0  2021            Kenine  103.02    67601.87
1  2021     Fajar Harapan   21.36    14732.88
2  2021      Damaran Baru  174.01    133971.1
3  2021  Pantan Pediangan  285.18   191159.97
4  2021   Bandar Lampahan   43.13     28048.9
5  2021           Rembune   64.88     48185.4
6  2021       Mude Benara   137.3     94597.1
7  2021          Bumi Ayu   79.07    59300.45
8  2021       Karang Jadi   85.71    64958.71
9  2021   Kampung Baru 76  360.64   234782.69


### Create manual calculation with excel
The formula have some issues due to difference in raw data and excel format

### For coffee_yield_pipeline.xlsx

In [2]:
# AUTO FETCH
import joblib
import pandas as pd

# Configuration
EXCEL_FILE = '../../code/coffee_yield_pipeline.xlsx'
SHEET_NAME='INFERENCE'
MODEL_PATH = '../result/best_model.pkl'
FEATURES_PATH = '../result/features.csv'

# Feature names in EXACT order (D44:D94)
FEATURE_NAMES = [
    'rainfall_mm','temperature_celsius','relative_humidity_percent',
    'soil_moisture_percent','wind_speed_10m','dtr_celsius','vpd_kpa',
    'net_solar_rad_kwh_m2','rainfall_mm_Q1','temperature_celsius_Q1',
    'relative_humidity_percent_Q1','soil_moisture_percent_Q1',
    'wind_speed_10m_Q1','dtr_celsius_Q1','vpd_kpa_Q1',
    'net_solar_rad_kwh_m2_Q1','rainfall_mm_Q2','temperature_celsius_Q2',
    'relative_humidity_percent_Q2','soil_moisture_percent_Q2',
    'wind_speed_10m_Q2','dtr_celsius_Q2','vpd_kpa_Q2',
    'net_solar_rad_kwh_m2_Q2','rainfall_mm_Q3','temperature_celsius_Q3',
    'relative_humidity_percent_Q3','soil_moisture_percent_Q3',
    'wind_speed_10m_Q3','dtr_celsius_Q3','vpd_kpa_Q3',
    'net_solar_rad_kwh_m2_Q3','rainfall_mm_Q4','temperature_celsius_Q4',
    'relative_humidity_percent_Q4','soil_moisture_percent_Q4',
    'wind_speed_10m_Q4','dtr_celsius_Q4','vpd_kpa_Q4',
    'net_solar_rad_kwh_m2_Q4','temperature_celsius_x_vpd_kpa',
    'temperature_celsius_x_soil_moisture_percent',
    'temperature_celsius_x_dtr_celsius',
    'temperature_celsius_x_rainfall_mm',
    'vpd_kpa_x_soil_moisture_percent','vpd_kpa_x_dtr_celsius',
    'vpd_kpa_x_rainfall_mm','soil_moisture_percent_x_dtr_celsius',
    'soil_moisture_percent_x_rainfall_mm',
    'dtr_celsius_x_rainfall_mm','prev_yield_kg_ha'
]

def main():
    model = joblib.load(MODEL_PATH)
    feats = pd.read_csv(FEATURES_PATH)['feature'].tolist()
    
    try:
        df = pd.read_excel(EXCEL_FILE, sheet_name=SHEET_NAME, header=None)
    except ValueError as e:
        print(f"{e}")
        return
    
    # Extract values from column D (index 3), rows 44-94 (index 43-93)
    data = pd.DataFrame([df.iloc[43:94, 3].tolist()], columns=FEATURE_NAMES)
    
    # Check for missing/NaN values
    missing = data.columns[data.isna().any()].tolist()
    if missing:
        print(f"Warning!!! Fill the excel first!")
        print(f"{len(missing)} missing features: {missing}")
        return
    
    print(f"log_yield = {model.predict(data.reindex(columns=feats))[0]:.6f}")
    
if __name__ == "__main__":
    main()

log_yield = 6.498081
